# JARVIS Wake-Word Training — «Джарвис»

Тренировка кастомной openWakeWord модели на русское слово «Джарвис».

**Требования:**
- Colab runtime с T4 GPU (Runtime → Change runtime type → T4 GPU)
- Подготовленный `dataset.zip` из `workspace/wake_samples/dataset.zip` (см. `docs/B8_WAKE_WORD_TRAINING.md`)

**Время:** 6-8 часов на T4. Бесплатный Colab отрубается через 12 часов — сохраняй промежуточные веса в Drive.

**Если этот notebook сломан** (openWakeWord API меняется) — открой [официальный automatic_model_training.ipynb](https://github.com/dscripka/openWakeWord/blob/main/notebooks/automatic_model_training.ipynb) и адаптируй под русский.

In [ ]:
# 1. Проверка GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('VRAM:', torch.cuda.get_device_properties(0).total_memory / 1e9, 'GB')
else:
    raise RuntimeError('GPU не активен. Runtime → Change runtime type → T4 GPU → Save → Connect.')

In [ ]:
# 2. Установка зависимостей
!pip install -q openwakeword onnxruntime-gpu soundfile librosa audiomentations
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu121
print('OK')

In [ ]:
# 3. Загрузка dataset.zip
# Вариант A — через браузер (для одноразового запуска):
from google.colab import files
print('Загрузи dataset.zip из workspace/wake_samples/dataset.zip:')
uploaded = files.upload()

# Вариант B — через Google Drive (раскомментируй если предпочитаешь):
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/jarvis-wake/dataset.zip /content/

In [ ]:
# 4. Распаковка и проверка
import zipfile, os
with zipfile.ZipFile('dataset.zip') as zf:
    zf.extractall('/content/wake_data')
pos = os.listdir('/content/wake_data/positive')
neg = os.listdir('/content/wake_data/negative')
print(f'positive: {len(pos)} файлов')
print(f'negative: {len(neg)} файлов')
assert len(pos) >= 50, 'нужно минимум 50 positive samples'
assert len(neg) >= 20, 'нужно минимум 20 negative samples'

In [ ]:
# 5. Клонирование репо с training pipeline
!git clone https://github.com/dscripka/openWakeWord.git /content/oww_repo
%cd /content/oww_repo
!pip install -e .

## Следующие шаги — официальный training notebook

Дальше тренировка идёт по [официальному notebook'у dscripka](https://github.com/dscripka/openWakeWord/blob/main/notebooks/automatic_model_training.ipynb).

Открой его в Colab и:
1. Замени `target_word = "hey jarvis"` на `target_word = "джарвис"`
2. В переменной `custom_positive_samples` укажи `/content/wake_data/positive`
3. В переменной `custom_negative_samples` укажи `/content/wake_data/negative`
4. Run All.

После завершения (6-8 часов) — финальная `.onnx` модель окажется в `/content/dzarvis_model/`.

In [ ]:
# 6. После завершения training — скачать финальную модель
import glob, os, shutil
onnx_files = glob.glob('/content/**/*.onnx', recursive=True)
# Фильтруем — только реальные wake-модели, не embedding/melspec
candidates = [f for f in onnx_files if 'dzarvis' in f.lower() or 'wake' in f.lower() or 'final' in f.lower()]
if not candidates:
    candidates = [f for f in onnx_files if 'embedding' not in f and 'melspec' not in f and 'silero' not in f]
print('Найденные ONNX-модели:')
for f in candidates:
    print(f'  {os.path.getsize(f)/1024:.1f} KB  {f}')

if candidates:
    final = max(candidates, key=os.path.getmtime)
    target = '/content/dzarvis.onnx'
    shutil.copy(final, target)
    print(f'\nГотово: {target}')
    from google.colab import files
    files.download(target)

## После скачивания `dzarvis.onnx`

Положи модель в репо JARVIS:
```
C:\Users\Staho\Documents\Claude\Projects\ДЖАРВИС (2)\jarvis\models\wake\dzarvis.onnx
```

Подробности в `docs/B8_WAKE_WORD_TRAINING.md` → раздел 4 (Интеграция).